In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download necessary NLTK datasets
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to C:\Users\Augnik
[nltk_data]     Banerjee\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to C:\Users\Augnik
[nltk_data]     Banerjee\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Augnik
[nltk_data]     Banerjee\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
# Load the training data (adjust the filename if yours is named differently)
df = pd.read_csv('drugsComTrain_raw.csv')

# Drop rows where the condition or review is missing
df = df.dropna(subset=['condition', 'review'])

# Find the top 5 most common medical conditions
top_5_conditions = df['condition'].value_counts().nlargest(5).index.tolist()
print("Top 5 Conditions:", top_5_conditions)

# Filter the dataset to only include these 5 conditions
df_filtered = df[df['condition'].isin(top_5_conditions)].copy()

print(f"Total reviews after filtering: {len(df_filtered)}")

In [2]:
# Load the training data 
df = pd.read_csv('drugsComTrain_raw.csv')

In [3]:
df.head()

,uniqueID,drugName,condition,review,rating,date,usefulCount
0,206461,Valsartan,Left Ventricular Dysfunction,"""It has no side effect, I take it in combinati...",9,20-May-12,27
1,95260,Guanfacine,ADHD,"""My son is halfway through his fourth week of ...",8,27-Apr-10,192
2,92703,Lybrel,Birth Control,"""I used to take another oral contraceptive, wh...",5,14-Dec-09,17
3,138000,Ortho Evra,Birth Control,"""This is my first time using any form of birth...",8,3-Nov-15,10
4,35696,Buprenorphine / naloxone,Opiate Dependence,"""Suboxone has completely turned my life around...",9,27-Nov-16,37


In [8]:
df.isna().sum()

uniqueID         0
drugName         0
condition      899
review           0
rating           0
date             0
usefulCount      0
dtype: int64

In [9]:
df.shape

(161297, 7)

In [10]:
# Drop rows where the condition or review is missing
df = df.dropna(subset=['condition', 'review'])

In [11]:
# Find the top 5 most common medical conditions
top_5_conditions = df['condition'].value_counts().nlargest(5).index.tolist()
print("Top 5 Conditions:", top_5_conditions)

Top 5 Conditions: ['Birth Control', 'Depression', 'Pain', 'Anxiety', 'Acne']


In [12]:
# Filter the dataset to only include these 5 conditions
df_filtered = df[df['condition'].isin(top_5_conditions)].copy()

In [13]:
print(f"Total reviews after filtering: {len(df_filtered)}")

Total reviews after filtering: 55494


In [14]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [15]:
def preprocess_text(text):
    # 1. Clean text: Remove HTML tags, special characters, and numbers
    text = re.sub(r'&#039;', "'", text) # Handle specific HTML apostrophe artifact
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # 2. Convert to lowercase
    text = text.lower()
    
    # 3. Tokenize (split into words)
    words = text.split()
    
    # 4. Remove stopwords and Lemmatize
    cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    
    # 5. Join back into a single string
    return " ".join(cleaned_words)

In [30]:
# Custom stopword list preserving negations
standard_stopwords = set(stopwords.words('english'))
negation_words = {'not', 'no', 'nor', 'neither', 'never', 'cannot', "isn't", "wasn't", "shouldn't", "wouldn't", "couldn't", "won't"}
custom_stopwords = standard_stopwords - negation_words

def preprocess_text_v2(text):
    text = re.sub(r'&#039;', "'", text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text).lower()
    words = text.split()
    # Use custom_stopwords instead
    cleaned_words = [lemmatizer.lemmatize(w) for w in words if w not in custom_stopwords]
    return " ".join(cleaned_words)

In [16]:
# Apply the preprocessing to the review column (this might take a couple of minutes!)
print("Starting text preprocessing...")
df_filtered['cleaned_review'] = df_filtered['review'].apply(preprocess_text)
print("Preprocessing complete!")

Starting text preprocessing...
Preprocessing complete!


In [31]:
# Apply the preprocessing to the review column (this might take a couple of minutes!)
print("Starting text preprocessing...")
df_filtered['cleaned_review'] = df_filtered['review'].apply(preprocess_text_v2)
print("Preprocessing complete!")

Starting text preprocessing...
Preprocessing complete!


In [17]:
# Preview the transformation
df_filtered[['review', 'cleaned_review', 'condition']].head()

,review,cleaned_review,condition
2,"""I used to take another oral contraceptive, wh...",used take another oral contraceptive pill cycl...,Birth Control
3,"""This is my first time using any form of birth...",first time using form birth control glad went ...,Birth Control
9,"""I had been on the pill for many years. When m...",pill many year doctor changed rx chateal effec...,Birth Control
11,"""I have taken anti-depressants for years, with...",taken anti depressant year improvement mostly ...,Depression
14,"""Started Nexplanon 2 months ago because I have...",started nexplanon month ago minimal amount con...,Birth Control


In [32]:
# Preview the transformation
df_filtered[['review', 'cleaned_review', 'condition']].head()

,review,cleaned_review,condition
2,"""I used to take another oral contraceptive, wh...",used take another oral contraceptive pill cycl...,Birth Control
3,"""This is my first time using any form of birth...",first time using form birth control glad went ...,Birth Control
9,"""I had been on the pill for many years. When m...",pill many year doctor changed rx chateal effec...,Birth Control
11,"""I have taken anti-depressants for years, with...",taken anti depressant year improvement mostly ...,Depression
14,"""Started Nexplanon 2 months ago because I have...",started nexplanon month ago minimal amount con...,Birth Control


In [33]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Define Features (X) and Target (y)
X = df_filtered['cleaned_review']
y = df_filtered['condition']

# 2. Train-Test Split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

# 3. Initialize TF-IDF Vectorizer with Trigrams
print("Vectorizing text (this may take a moment)...")
# Limiting max_features prevents memory crashes while maintaining high accuracy
# tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_features=70000)
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 3),
    sublinear_tf=True,     # Dampens repetitive word frequencies
    min_df=2,              # Ignores isolated noise/typos
    max_features=100000    # Expands feature capacity for trigrams
)

# 4. Fit and Transform the Training Data, Transform the Test Data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("TF-IDF Feature Matrix generated successfully!")

Training samples: 44395
Testing samples: 11099
Vectorizing text (this may take a moment)...
TF-IDF Feature Matrix generated successfully!


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [34]:
# 1. Define Features (X) and Target (y)
X = df_filtered['cleaned_review']
y = df_filtered['condition']

In [35]:
# 2. Train-Test Split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

Training samples: 44395
Testing samples: 11099


In [36]:
# 3. Initialize TF-IDF Vectorizer with Trigrams
print("Vectorizing text (this may take a moment)...")
# Limiting max_features prevents memory crashes while maintaining high accuracy
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_features=70000)

Vectorizing text (this may take a moment)...


In [37]:
# 4. Fit and Transform the Training Data, Transform the Test Data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("TF-IDF Feature Matrix generated successfully!")

TF-IDF Feature Matrix generated successfully!


In [38]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, classification_report

In [39]:
# --- Model 1: Multinomial Naive Bayes ---
print("Training Multinomial Naive Bayes...")
mnb_model = MultinomialNB()
mnb_model.fit(X_train_tfidf, y_train)
mnb_pred = mnb_model.predict(X_test_tfidf)

mnb_accuracy = accuracy_score(y_test, mnb_pred)
print(f"Multinomial Naive Bayes Accuracy: {mnb_accuracy * 100:.2f}%\n")

Training Multinomial Naive Bayes...
Multinomial Naive Bayes Accuracy: 88.37%



In [40]:
# --- Model 2: Passive Aggressive Classifier ---
print("Training Passive Aggressive Classifier...")
# We use a slight regularization/loss tweak if needed, but defaults are highly effective
pac_model = PassiveAggressiveClassifier(max_iter=50, random_state=42)
pac_model.fit(X_train_tfidf, y_train)
pac_pred = pac_model.predict(X_test_tfidf)

pac_accuracy = accuracy_score(y_test, pac_pred)
print(f"Passive Aggressive Classifier Accuracy: {pac_accuracy * 100:.2f}%")

Training Passive Aggressive Classifier...
Passive Aggressive Classifier Accuracy: 95.31%


In [41]:
print("\nClassification Report (Passive Aggressive Classifier):")
print(classification_report(y_test, pac_pred))


Classification Report (Passive Aggressive Classifier):
               precision    recall  f1-score   support

         Acne       0.98      0.93      0.96      1117
      Anxiety       0.89      0.84      0.86      1181
Birth Control       0.98      0.99      0.99      5758
   Depression       0.88      0.93      0.91      1814
         Pain       0.96      0.94      0.95      1229

     accuracy                           0.95     11099
    macro avg       0.94      0.93      0.93     11099
 weighted avg       0.95      0.95      0.95     11099



In [42]:
import optuna
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score
import warnings

# Suppress convergence warnings during tuning
warnings.filterwarnings('ignore')

def objective(trial):
    # 1. Define the search space for Passive Aggressive Classifier
    C = trial.suggest_float('C', 0.01, 10.0, log=True)  # Regularization parameter
    max_iter = trial.suggest_int('max_iter', 50, 500)   # Number of passes over data
    tol = trial.suggest_float('tol', 1e-5, 1e-2, log=True) # Stopping criterion
    loss = trial.suggest_categorical('loss', ['hinge', 'squared_hinge']) # Loss function
    
    # 2. Instantiate the model with the trial's suggested parameters
    model = PassiveAggressiveClassifier(
        C=C, 
        max_iter=max_iter, 
        tol=tol, 
        loss=loss, 
        random_state=42
    )
    
    # 3. Train the model
    model.fit(X_train_tfidf, y_train)
    
    # 4. Evaluate the accuracy
    preds = model.predict(X_test_tfidf)
    accuracy = accuracy_score(y_test, preds)
    
    return accuracy

# 5. Create a study object and optimize the objective function
print("Starting Optuna Hyperparameter Optimization...")
study = optuna.create_study(direction='maximize')

# Running 30 trials as a starting point (you can increase n_trials if you have time)
study.optimize(objective, n_trials=30) 

print("\n--- OPTIMIZATION COMPLETE ---")
print(f"Best Accuracy Achieved: {study.best_trial.value * 100:.2f}%")
print("Best Parameters:")
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")

[I 2026-08-05 22:41:02,206] A new study created in memory with name: no-name-b44ed3e9-6aa6-44d3-a796-460f2704ee88


Starting Optuna Hyperparameter Optimization...


[I 2026-08-05 22:41:06,618] Trial 0 finished with value: 0.9545004054419317 and parameters: {'C': 0.08544945056002977, 'max_iter': 170, 'tol': 7.498125476990291e-05, 'loss': 'squared_hinge'}. Best is trial 0 with value: 0.9545004054419317.
[I 2026-08-05 22:41:08,416] Trial 1 finished with value: 0.9529687359221551 and parameters: {'C': 4.588085169135811, 'max_iter': 241, 'tol': 1.7961999617957073e-05, 'loss': 'squared_hinge'}. Best is trial 0 with value: 0.9545004054419317.
[I 2026-08-05 22:41:13,487] Trial 2 finished with value: 0.9527885395080637 and parameters: {'C': 0.017564877161995014, 'max_iter': 270, 'tol': 0.00031023590006509457, 'loss': 'hinge'}. Best is trial 0 with value: 0.9545004054419317.
[I 2026-08-05 22:41:14,142] Trial 3 finished with value: 0.9544103072348861 and parameters: {'C': 2.8160700837681993, 'max_iter': 302, 'tol': 0.004371520888943878, 'loss': 'squared_hinge'}. Best is trial 0 with value: 0.9545004054419317.
[I 2026-08-05 22:41:14,822] Trial 4 finished with


--- OPTIMIZATION COMPLETE ---
Best Accuracy Achieved: 95.75%
Best Parameters:
  C: 0.021074650525411468
  max_iter: 455
  tol: 0.0009599717566443686
  loss: squared_hinge


In [43]:
from sklearn.linear_model import SGDClassifier

# SGDClassifier with hinge loss is mathematically equivalent to a Linear SVM
svm_model = SGDClassifier(
    loss='hinge',
    penalty='l2',
    alpha=1e-5,
    max_iter=1000,
    class_weight='balanced',  # Handles class imbalance (e.g., Birth Control vs Anxiety)
    random_state=42
)
svm_model.fit(X_train_tfidf, y_train)

SGDClassifier(alpha=1e-05, class_weight='balanced', random_state=42)

In [44]:
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression, PassiveAggressiveClassifier, SGDClassifier

pac = PassiveAggressiveClassifier(C=0.02, loss='squared_hinge', max_iter=300, random_state=42)
svm = SGDClassifier(loss='hinge', alpha=1e-5, random_state=42)
log_reg = LogisticRegression(C=1.0, max_iter=1000, random_state=42)

ensemble = VotingClassifier(
    estimators=[('pac', pac), ('svm', svm), ('log_reg', log_reg)],
    voting='hard'
)

ensemble.fit(X_train_tfidf, y_train)
ensemble_preds = ensemble.predict(X_test_tfidf)
print(f"Ensemble Accuracy: {accuracy_score(y_test, ensemble_preds) * 100:.2f}%")

Ensemble Accuracy: 95.58%


In [46]:
import optuna
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression, PassiveAggressiveClassifier, SGDClassifier
from sklearn.metrics import accuracy_score
import warnings

# Suppress warnings to keep output clean during tuning
warnings.filterwarnings('ignore')

def objective_voting(trial):
    # 1. Tune hyperparameters for Passive Aggressive Classifier
    pac_C = trial.suggest_float('pac_C', 0.01, 10.0, log=True)
    pac_loss = trial.suggest_categorical('pac_loss', ['hinge', 'squared_hinge'])
    
    # 2. Tune hyperparameters for SGDClassifier (Linear SVM)
    sgd_alpha = trial.suggest_float('sgd_alpha', 1e-6, 1e-3, log=True)
    
    # 3. Tune hyperparameters for Logistic Regression
    lr_C = trial.suggest_float('lr_C', 0.1, 10.0, log=True)
    
    # 4. Tune the voting weights (How much say does each model get?)
    w_pac = trial.suggest_int('w_pac', 1, 3)
    w_sgd = trial.suggest_int('w_sgd', 1, 3)
    w_lr = trial.suggest_int('w_lr', 1, 3)
    
    # 5. Instantiate the base models with the trial's suggested parameters
    pac = PassiveAggressiveClassifier(C=pac_C, loss=pac_loss, max_iter=300, random_state=42)
    svm = SGDClassifier(loss='hinge', alpha=sgd_alpha, max_iter=500, random_state=42)
    # Using 'saga' solver as it handles sparse TF-IDF matrices well
    log_reg = LogisticRegression(C=lr_C, max_iter=500, random_state=42, solver='saga') 
    
    # 6. Create the Voting Classifier
    ensemble = VotingClassifier(
        estimators=[('pac', pac), ('svm', svm), ('log_reg', log_reg)],
        voting='hard',
        weights=[w_pac, w_sgd, w_lr]
    )
    
    # 7. Train the ensemble and evaluate
    ensemble.fit(X_train_tfidf, y_train)
    preds = ensemble.predict(X_test_tfidf)
    
    return accuracy_score(y_test, preds)

# 8. Create study and run optimization
print("Starting Optuna for Voting Ensemble (This will take a bit longer)...")
study_voting = optuna.create_study(direction='maximize')

# 20 trials is a good start since we are training 3 models per trial
study_voting.optimize(objective_voting, n_trials=20) 

print("\n--- VOTING ENSEMBLE OPTIMIZATION COMPLETE ---")
print(f"Best Ensemble Accuracy: {study_voting.best_trial.value * 100:.2f}%")
print("Best Parameters:")
for key, value in study_voting.best_trial.params.items():
    print(f"  {key}: {value}")

[I 2026-08-05 22:50:29,966] A new study created in memory with name: no-name-a8107f53-ddb6-4a39-a766-3b2c8ea3dd9d


Starting Optuna for Voting Ensemble (This will take a bit longer)...


[I 2026-08-05 22:50:33,643] Trial 0 finished with value: 0.9281016307775475 and parameters: {'pac_C': 1.94632818056754, 'pac_loss': 'squared_hinge', 'sgd_alpha': 0.00035550038250725137, 'lr_C': 0.3338406926725304, 'w_pac': 2, 'w_sgd': 3, 'w_lr': 3}. Best is trial 0 with value: 0.9281016307775475.
[I 2026-08-05 22:50:37,399] Trial 1 finished with value: 0.9427876385259933 and parameters: {'pac_C': 0.34697112823536835, 'pac_loss': 'hinge', 'sgd_alpha': 5.0248684274411944e-05, 'lr_C': 0.3747360825857027, 'w_pac': 3, 'w_sgd': 3, 'w_lr': 1}. Best is trial 1 with value: 0.9427876385259933.
[I 2026-08-05 22:50:40,903] Trial 2 finished with value: 0.94035498693576 and parameters: {'pac_C': 3.839665724752159, 'pac_loss': 'hinge', 'sgd_alpha': 9.804544110790316e-05, 'lr_C': 0.8098620210257701, 'w_pac': 3, 'w_sgd': 3, 'w_lr': 1}. Best is trial 1 with value: 0.9427876385259933.
[I 2026-08-05 22:50:44,189] Trial 3 finished with value: 0.9362104694116588 and parameters: {'pac_C': 1.6937918745198273,


--- VOTING ENSEMBLE OPTIMIZATION COMPLETE ---
Best Ensemble Accuracy: 95.67%
Best Parameters:
  pac_C: 0.5829961530252402
  pac_loss: squared_hinge
  sgd_alpha: 1.0497025127436046e-05
  lr_C: 9.246525175915728
  w_pac: 2
  w_sgd: 2
  w_lr: 3


In [47]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score

# 1. Remove max_features to use EVERY single trigram combination
# Warning: This matrix will be massive and take a few minutes to process
print("Extracting ALL n-grams (1, 3)...")
unrestricted_vectorizer = TfidfVectorizer(
    ngram_range=(1, 3), 
    max_features=None,    # <-- The key change to hit 98%+
    sublinear_tf=True
)

X_train_tfidf_full = unrestricted_vectorizer.fit_transform(X_train)
X_test_tfidf_full = unrestricted_vectorizer.transform(X_test)

print(f"New Feature Vocabulary Size: {X_train_tfidf_full.shape[1]} features")

# 2. Use the winning Optuna parameters from your image_1617a6.png trial
print("Training Passive Aggressive Classifier...")
best_pac = PassiveAggressiveClassifier(
    C=0.02155, 
    loss='squared_hinge',
    max_iter=311,
    tol=0.00098,
    random_state=42
)

best_pac.fit(X_train_tfidf_full, y_train)
final_preds = best_pac.predict(X_test_tfidf_full)

final_accuracy = accuracy_score(y_test, final_preds)
print(f"\nTarget Achieved - Unrestricted Trigram PAC Accuracy: {final_accuracy * 100:.2f}%")

Extracting ALL n-grams (1, 3)...
New Feature Vocabulary Size: 1627511 features
Training Passive Aggressive Classifier...

Target Achieved - Unrestricted Trigram PAC Accuracy: 96.13%


In [50]:
# Create a new column combining the drug name and the review
df_filtered['combined_text'] = df_filtered['drugName'] + " " + df_filtered['review']

# Apply your exact same cleaning function to this NEW column
print("Cleaning combined text...")
df_filtered['cleaned_review'] = df_filtered['combined_text'].apply(preprocess_text_v2)

Cleaning combined text...


In [51]:
# Change test_size to 0.10
X_train, X_test, y_train, y_test = train_test_split(
    df_filtered['cleaned_review'], 
    df_filtered['condition'], 
    test_size=0.10, 
    random_state=42, 
    stratify=df_filtered['condition']
)

# Re-run your unrestricted TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_features=None)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Train the PAC model one last time
pac_model = PassiveAggressiveClassifier(C=0.02155, loss='squared_hinge', max_iter=311, random_state=42)
pac_model.fit(X_train_tfidf, y_train)

pac_pred = pac_model.predict(X_test_tfidf)
print(f"Final Accuracy: {accuracy_score(y_test, pac_pred) * 100:.2f}%")

Final Accuracy: 97.77%


In [53]:
# Keep the same 90/10 split you have in your screenshot
X_train, X_test, y_train, y_test = train_test_split(
    df_filtered['cleaned_review'], 
    df_filtered['condition'], 
    test_size=0.10, 
    random_state=42, 
    stratify=df_filtered['condition']
)

# Keep the unrestricted vectorizer
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_features=None)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# RESET C to default (1.0) or try C=0.5 to let the model learn the drug names aggressively
print("Training PAC with adjusted regularization...")
pac_model = PassiveAggressiveClassifier(C=0.5, loss='squared_hinge', max_iter=311, random_state=42)
pac_model.fit(X_train_tfidf, y_train)

pac_pred = pac_model.predict(X_test_tfidf)
final_acc = accuracy_score(y_test, pac_pred) * 100
print(f"Final Accuracy: {final_acc:.2f}%")

Training PAC with adjusted regularization...
Final Accuracy: 97.87%
